# NEURAL-RMF — Pre-ictal EEG Detection Demo

**Real-time pre-ictal detection for epilepsy monitoring.**

This notebook demonstrates the full pipeline on a CHB-MIT EDF file:
1. Install the library
2. Download a public EDF recording
3. Run calibration (first 8 min) + streaming detection
4. Inspect the alert timeline
5. Visualize the alert semaphore

---
> **Validated performance (4 frontal-temporal channels: F7, T7, F8, T8)**
>
> | Dataset | Detection rate | Mean lead time |
> |---|---|---|
> | CHB-MIT Scalp EEG (pediatric) | 100 % | 74.5 min |
> | Siena Scalp EEG (adult) | 97 % | 89.5 min |
>
> False-alarm rate ≤ 0.5 % / hour at `rojo` level.

In [ ]:
# ── Cell 1 — Install ──────────────────────────────────────────────────────────
# Run once per Colab session.
!pip install git+https://github.com/Gusmal02/NEURAL-RMF.git --quiet

# Verify installation
import neural_rmf
print(f"neural-rmf version: {neural_rmf.__version__}")

In [ ]:
# ── Cell 2 — Download a public EDF from CHB-MIT (PhysioNet) ──────────────────
# CHB-MIT is a public dataset of pediatric scalp EEG recordings with annotated seizures.
# Reference: Shoeb, A. et al. (2010). PhysioNet.

import os, urllib.request

EDF_URL  = "https://physionet.org/files/chbmit/1.0.0/chb01/chb01_03.edf?download"
EDF_PATH = "chb01_03.edf"
ANN_URL  = "https://physionet.org/files/chbmit/1.0.0/chb01/chb01-summary.txt?download"
ANN_PATH = "chb01-summary.txt"

if not os.path.exists(EDF_PATH):
    print("Downloading EDF (~35 MB)…")
    urllib.request.urlretrieve(EDF_URL, EDF_PATH)
    urllib.request.urlretrieve(ANN_URL, ANN_PATH)
    print("Done.")
else:
    print("EDF already present.")

# Print seizure annotation for this file
with open(ANN_PATH) as f:
    txt = f.read()
# Extract block for chb01_03
start = txt.find("chb01_03.edf")
end   = txt.find("\n\n", start)
print("\n--- Annotation ---")
print(txt[start:end].strip())

In [ ]:
# ── Cell 3 — Run the pipeline ─────────────────────────────────────────────────
# run_edf():
#   • Uses the first 8 minutes as intra-session calibration baseline
#   • Processes the remaining recording in 30-second windows
#   • Returns one dict per window with novelty metrics and the three-level alert

from neural_rmf import run_edf

CHANNELS = ["F7", "T7", "F8", "T8"]   # diadem-style frontal-temporal setup

print("Running pipeline…")
results = run_edf(EDF_PATH, channels=CHANNELS)
print(f"Processed {len(results)} windows.")

# Preview first 5 windows
print("\nt_min(s)  t_max(s)  novelty_max  novelty_col  estado")
print("-" * 58)
for w in results[:5]:
    print(f"{w['t_min']:8.1f}  {w['t_max']:8.1f}  "
          f"{w['novelty_max']:.4f}       {w['novelty_col']:.4f}       "
          f"{w['estado']}")

In [ ]:
# ── Cell 4 — Inspect the alert timeline ──────────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)

# Seizure onset in chb01_03: ~2996 s (from annotation)
SEIZURE_ONSET_SEC = 2996

# Find first naranja / rojo window
alerts = df[df["estado"].isin(["naranja", "rojo"])]
if len(alerts):
    first_alert = alerts.iloc[0]
    lead_min = (SEIZURE_ONSET_SEC - first_alert["t_min"]) / 60
    print(f"First alert   : {first_alert['estado'].upper()}")
    print(f"Alert time    : {first_alert['t_min']:.0f} s ({first_alert['t_min']/60:.1f} min)")
    print(f"Seizure onset : {SEIZURE_ONSET_SEC} s ({SEIZURE_ONSET_SEC/60:.1f} min)")
    print(f"Lead time     : {lead_min:.1f} min before onset")
else:
    print("No alerts detected in this segment.")

# State distribution
print("\nAlert distribution:")
print(df["estado"].value_counts().to_string())

In [ ]:
# ── Cell 5 — Visualize ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

COLOR = {"verde": "#2ECC71", "naranja": "#F39C12", "rojo": "#E74C3C"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle("NEURAL-RMF — Pre-ictal detection: chb01_03", fontsize=13, fontweight="bold")

t = df["t_min"] / 60   # minutes

# ── Top: novelty curves ──
ax1.plot(t, df["novelty_max"], lw=1.2, color="#2980B9", label="novelty_max")
ax1.plot(t, df["novelty_col"], lw=1.2, color="#8E44AD", alpha=0.8, label="novelty_col")
ax1.axvline(SEIZURE_ONSET_SEC / 60, color="#E74C3C", lw=1.5, ls="--", label="seizure onset")
# P80 calibration threshold
if "umbral" in df.columns:
    ax1.axhline(df["umbral"].iloc[0], color="#F39C12", lw=1, ls=":", label="P80 threshold")
ax1.set_ylabel("Novelty", fontsize=10)
ax1.legend(fontsize=9, loc="upper left")
ax1.grid(True, alpha=0.3)

# ── Bottom: semaphore strip ──
for _, row in df.iterrows():
    ax2.barh(0, row["t_max"] / 60 - row["t_min"] / 60,
             left=row["t_min"] / 60, height=1,
             color=COLOR.get(row["estado"], "#95A5A6"), alpha=0.85)
ax2.axvline(SEIZURE_ONSET_SEC / 60, color="#E74C3C", lw=1.5, ls="--")
ax2.set_yticks([])
ax2.set_xlabel("Time (minutes)", fontsize=10)
ax2.set_ylabel("Alert", fontsize=10)
patches = [mpatches.Patch(color=c, label=l) for l, c in COLOR.items()]
ax2.legend(handles=patches, fontsize=9, loc="upper left")
ax2.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("neural_rmf_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to neural_rmf_demo.png")